### Setup and import

In [ ]:
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from datetime import datetime

# Specify the directory where you want to save the files
output_directory = 'yourdirectory'

df_final = pd.read_excel(f'{output_directory}yourfilename.xlsx')

### Frailty Index

#### 1. select every variable that measures a health problem

In [ ]:
df_frailty = df_final.copy()
columns_to_keep =[ 'unique_episode',
                  'age',
 'hypertPathol',
 'mi',
 'chf',
 'pvd',
 'cvaOrTia',
 'dementia',
 'copd',
 'ctd',
 'pud',
 'hemiplegia',
 'cdk',
 'leukemia',
 'lymphoma',
 'liverDisease',
 'diabetesSeverity',
 'solidTumor',
 'aids',
 'therapycount',
 'adm_bmi',
 'adm_SBP',
 'adm_DBP',
 'adm_heartRate',
 'adm_respiratoryRate',
 'adm_bodyTemp',
 'adm_SpO2',
 'adm_FiO2',
 'adm_fragileSubject',
 'adm_dependency',
 'adm_homeAssistance',
 'adm_karnScale',
 'Hb_g_dl',
 'WBC_10_9l',
 'PLT_10_9l',
 'AST_U_L',
 'ALT_U_L',
 'LDH_U_L',
 'crea_mg_dl',
 'totProt_g_L',
 'Albumin_perc',
 'CRP_mg_l',
 'Na_mmol_L',
 'K_mmol_L',
 'badm_feeding',
 'badm_bathing',
 'badm_hygiene',
 'badm_dressing',
 'badm_bowel',
 'badm_bladder',
 'badm_toilet',
 'badm_transfers',
 'badm_walking',
 'badm_stairs']
df_frailty = df_frailty[columns_to_keep]
df_frailty

#### 2. exclude variables with more than 5% missingvalues

In [ ]:
# show percentage of missing values and sort them in descending order showing only those above 15%
print((df_frailty.isna().mean().sort_values(ascending=False) * 100).loc[lambda x: x > 15])

# exclude variables with more than 5% missing values
df_frailty = df_frailty.loc[:, df_frailty.isna().mean() < 0.16]

#### 3. recode the responses to 0 (nodeficit) through 1 (deficit)

In [ ]:
# Apply the conditions to create 0-1 columns, preserving NaN values

# MEDICAL HISTORY
df_frailty['frailty_polypharmacy'] = np.where(df_frailty['therapycount'].isna(), np.nan,
                          (df_frailty['therapycount'] >= 5).astype(int))
df_frailty['frailty_hypertension'] = np.where(df_frailty['hypertPathol'].isna(), np.nan, 
                                  (df_frailty['hypertPathol'] != "0").astype(int))
df_frailty['frailty_diabetes'] = np.where(
    df_frailty['diabetesSeverity'].isna(),
    np.nan,
    df_frailty['diabetesSeverity'].map({
        0: 0,
        1: 0.5,
        2: 1
    })
)
df_frailty['frailty_ckd'] = np.where(df_frailty['cdk'].isna(), np.nan, 
                           (df_frailty['cdk'] != 0).astype(int))
df_frailty['frailty_ihd'] = np.where(df_frailty['mi'].isna(), np.nan,
                            (df_frailty['mi'] != 0).astype(int))
df_frailty['frailty_chf'] = np.where(df_frailty['chf'].isna(), np.nan,
                            (df_frailty['chf'] != 0).astype(int))
df_frailty['frailty_pvd'] = np.where(df_frailty['pvd'].isna(), np.nan,
                            (df_frailty['pvd'] != 0).astype(int))
df_frailty['frailty_copd'] = np.where(df_frailty['copd'].isna(), np.nan,
                            (df_frailty['copd'] != 0).astype(int))
df_frailty['frailty_liver'] = np.where(
    df_frailty['liverDisease'].isna(),
    np.nan,
    df_frailty['liverDisease'].map({
        0: 0,
        1: 0.5,
        3: 1
    })
)
df_frailty['frailty_cancer'] = np.where(df_frailty['solidTumor'].isna(), np.nan,
                                 ((df_frailty['solidTumor'] != 0) | 
                                  (df_frailty['lymphoma'] != 0) | 
                                  (df_frailty['leukemia'] != 0)).astype(int))             # Combine solid tumor, lymphoma, and leukemia into one column
df_frailty['frailty_dementia'] = np.where(df_frailty['dementia'].isna(), np.nan,
                                      (df_frailty['dementia'] != 0).astype(int))
df_frailty['frailty_cva'] = np.where(df_frailty['cvaOrTia'].isna(), np.nan,
                                    (df_frailty['cvaOrTia'] != 0).astype(int))

# VITALS
df_frailty['frailty_bmi'] = np.where(df_frailty['adm_bmi'].isna(), np.nan, 
                                     ((df_frailty['adm_bmi'] < 22) | (df_frailty['adm_bmi'] > 27) & (df_frailty['age'] >= 65) |
                                      (df_frailty['adm_bmi'] < 18.5) | (df_frailty['adm_bmi'] > 30) & (df_frailty['age'] < 65)).astype(int))
##df_frailty['frailty_bodyTemp'] = np.where(df_frailty['adm_bodyTemp'].isna(), np.nan, 
#                             ((df_frailty['adm_bodyTemp'] < 36) | (df_frailty['adm_bodyTemp'] > 37.7)).astype(int))
df_frailty['frailty_SBP'] = np.where(df_frailty['adm_SBP'].isna(), np.nan, 
                          ((df_frailty['adm_SBP'] > 130) | (df_frailty['adm_SBP'] < 90)).astype(int))
df_frailty['frailty_DBP'] = np.where(df_frailty['adm_DBP'].isna(), np.nan, 
                       ((df_frailty['adm_DBP'] > 85) | (df_frailty['adm_DBP'] < 60)).astype(int))
df_frailty['frailty_SpO2'] = np.where(df_frailty['adm_SpO2'].isna(), np.nan, 
                            (df_frailty['adm_SpO2'] < 94).astype(int))
df_frailty['frailty_heartRate'] = np.where(df_frailty['adm_heartRate'].isna(), np.nan, 
                           ((df_frailty['adm_heartRate'] > 100) | (df_frailty['adm_heartRate'] < 60)).astype(int))


# MISCELLANEOUS
df_frailty['frailty_dependency'] = np.where(df_frailty['adm_dependency'].isna(), np.nan, 
                                       (df_frailty['adm_dependency'] != 'None').astype(int))
df_frailty['frailty_homeAssistance'] = np.where(df_frailty['adm_homeAssistance'].isna(), np.nan, 
                                  (df_frailty['adm_homeAssistance'] != 'None').astype(int))
#df_frailty['adm_karnScale'] = df_frailty['adm_karnScale'].astype(float)  # make karnofsky numerical
#df_frailty['frailty_karnofsky'] = np.where(df_frailty['adm_karnScale'].isna(), np.nan,
#                                        (df_frailty['adm_karnScale'] < 100).astype(int))

# REMOVED BARTHEL; CHECK NEXT CELL

# LABS
df_frailty['frailty_sodium'] = np.where(df_frailty['Na_mmol_L'].isna(), np.nan, 
                         ((df_frailty['Na_mmol_L'] < 135) | (df_frailty['Na_mmol_L'] > 145)).astype(int))
df_frailty['frailty_CRP'] = np.where(df_frailty['CRP_mg_l'].isna(), np.nan, 
                      (df_frailty['CRP_mg_l'] > 20).astype(int))
df_frailty['frailty_Hb'] = np.where(df_frailty['Hb_g_dl'].isna(), np.nan, 
                             (df_frailty['Hb_g_dl'] < 12.5).astype(int))
df_frailty['frailty_WBC'] = np.where(df_frailty['WBC_10_9l'].isna(), np.nan, 
                                    ((df_frailty['WBC_10_9l'] < 4) | (df_frailty['WBC_10_9l'] > 10)).astype(int))
df_frailty['frailty_PLT'] = np.where(df_frailty['PLT_10_9l'].isna(), np.nan, 
                         ((df_frailty['PLT_10_9l'] < 150) | (df_frailty['PLT_10_9l'] > 400)).astype(int))
df_frailty['frailty_AST'] = np.where(df_frailty['AST_U_L'].isna(), np.nan, 
                         ((df_frailty['AST_U_L'] > 70)).astype(int))
df_frailty['frailty_ALT'] = np.where(df_frailty['ALT_U_L'].isna(), np.nan, 
                         ((df_frailty['ALT_U_L'] > 70)).astype(int))
df_frailty['frailty_LDH'] = np.where(df_frailty['LDH_U_L'].isna(), np.nan, 
                         ((df_frailty['LDH_U_L'] > 360)).astype(int))
df_frailty['frailty_Crea'] = np.where(df_frailty['crea_mg_dl'].isna(), np.nan,
                          ((df_frailty['crea_mg_dl'] > 1.3)).astype(int))
df_frailty['frailty_potassium'] = np.where(df_frailty['K_mmol_L'].isna(), np.nan, 
                            ((df_frailty['K_mmol_L'] < 3.5) | (df_frailty['K_mmol_L'] > 5)).astype(int))

# Remove columns that have been transformed into frailty indicators
columns_to_remove = ['hypertPathol',
 'mi',
 'chf',
 'pvd',
 'cvaOrTia',
 'dementia',
 'copd',
 'ctd',
 'pud',
 'hemiplegia',
 'cdk',
 'leukemia',
 'lymphoma',
 'liverDisease',
 'diabetesSeverity',
 'solidTumor',
 'aids',
 'therapycount',
 'adm_bmi',
 'adm_SBP',
 'adm_DBP',
 'adm_heartRate',
# 'adm_respiratoryRate',
# 'adm_bodyTemp',
 'adm_SpO2',
# 'adm_FiO2',
# 'adm_fragileSubject',
 'adm_dependency',
 'adm_homeAssistance',
# 'adm_karnScale',
 'Hb_g_dl',
 'WBC_10_9l',
 'PLT_10_9l',
 'AST_U_L',
 'ALT_U_L',
 'LDH_U_L',
 'crea_mg_dl',
# 'totProt_g_L',
# 'Albumin_perc',
 'CRP_mg_l',
 'Na_mmol_L',
 'K_mmol_L',
# 'badm_feeding',
# 'badm_bathing',
# 'badm_hygiene',
# 'badm_dressing',
# 'badm_bowel',
# 'badm_bladder',
# 'badm_toilet',
# 'badm_transfers',
# 'badm_walking',
# 'badm_stairs'
]
df_frailty.drop(columns=columns_to_remove, inplace=True)

#### 4. exclude variables when coded deficits are too rare (< 1%) or too common (> 80%)

In [ ]:
# evaluate counts of deficits to exclude patients with deficits that are either too rare (<1%) or too common (>80%)

# Identify all engineered frailty deficit columns
frailty_cols = [col for col in df_frailty.columns if col.startswith('frailty_')]

# Deficit prevalence: proportion of non-missing values that are != 0
# (this correctly handles ordinal variables like frailty_diabetes and frailty_liver)
deficit_prevalence = pd.Series({
    col: (df_frailty[col] != 0).sum() / df_frailty[col].notna().sum() * 100
    for col in frailty_cols
}).sort_values()

print(deficit_prevalence)

# Flag deficits that are too rare (<1%) or too common (>80%)
too_rare = deficit_prevalence[deficit_prevalence < 1].index.tolist()
too_common = deficit_prevalence[deficit_prevalence > 80].index.tolist()

print(f"\nToo rare (<1%): {too_rare}")
print(f"Too common (>80%): {too_common}")

# Drop the flagged columns from df_frailty
df_frailty = df_frailty.drop(columns=too_rare + too_common)

#### 5. Screen the coded variables for association with age and drop those not associated, unless known from literature or other validated frailty indices

In [ ]:
frailty_cols = [col for col in df_frailty.columns if col.startswith('frailty_')]

df_frailty['age_rounded'] = df_frailty['age'].round(0)

n_cols = 4
n_rows = int(np.ceil(len(frailty_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(frailty_cols):
    age_means = df_frailty.groupby('age_rounded')[col].mean().dropna()

    axes[i].scatter(age_means.index, age_means.values, s=15, alpha=0.7)

    # Add a linear trend line to visualize the age relationship
    if len(age_means) > 1:
        slope, intercept = np.polyfit(age_means.index, age_means.values, 1)
        trend_line = slope * age_means.index + intercept
        axes[i].plot(age_means.index, trend_line, color='red', linewidth=1.5)

    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('Age')
    axes[i].set_ylabel('Mean / Proportion with deficit')

# Remove any unused subplot axes
for j in range(len(frailty_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
columns_to_remove = [
    'frailty_ALT',
    'frailty_LDH',
    'frailty_AST',
    'frailty_ALT',
    'frailty_PLT',
    'frailty_DBP',
    'frailty_heartRate',
    'frailty_CRP',
    'frailty_bmi',
    'frailty_liver'    
]
df_frailty.drop(columns=columns_to_remove, inplace=True)

#### 6. Screen the coded variables for correlation with each other.

In [ ]:
# Screen frailty deficit variables for near-duplicate correlations and drop those with |r| > 0.95
frailty_cols = [col for col in df_frailty.columns if col.startswith('frailty_')]

from scipy.stats import spearmanr

# Calculate the Spearman's correlation and p-values
corr, pval = spearmanr(df_frailty[frailty_cols], nan_policy='omit')

# Convert to DataFrames for easier handling
corr_df = pd.DataFrame(corr, index=frailty_cols, columns=frailty_cols)
pval_df = pd.DataFrame(pval, index=frailty_cols, columns=frailty_cols)

# Prepare annotation matrix: show both correlation (rounded to 2 decimals) and asterisk for p < 0.05
annot = corr_df.round(2).astype(str) + np.where(pval_df < 0.05, "*", "")

plt.figure(figsize=(27, 22))
sns.heatmap(
    corr_df,
    annot=annot,
    fmt='s',
    cmap='coolwarm',
    xticklabels=True,
    yticklabels=True,
    linewidths=0.5,
    linecolor='gray',
    annot_kws={"fontsize": 13}  # Set annotation font size here
)
plt.xticks(fontsize=22, rotation=90)
plt.yticks(fontsize=22)
plt.xlabel('', fontsize=18)
plt.ylabel('', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
# Screen frailty deficit variables for near-duplicate correlations and drop those with |r| > 0.95
frailty_cols = [col for col in df_frailty.columns if col.startswith('frailty_')]

# Use only the engineered frailty columns
corr = df_frailty[frailty_cols].corr(numeric_only=True)

# Identify pairs with absolute correlation above threshold
to_drop = set()

for i, col in enumerate(corr.columns):
	for other in corr.columns[i + 1:]:
		r = corr.loc[col, other]
		if pd.notna(r) and abs(r) > 0.95:
			# Keep the first column encountered and drop the later one
			to_drop.add(other)

# Show what will be removed
print(f"Highly correlated columns to drop: {sorted(to_drop)}")

# Apply removal
df_frailty = df_frailty.drop(columns=sorted(to_drop))

#### 7. Count retained variables.

In [ ]:
frailty_cols = [col for col in df_frailty.columns if col.startswith('frailty_')]
print(f'Total retained columns: {len(frailty_cols)}')

#### 8a Frailty WITH Past Medical History (41 variables)

In [ ]:
df_frailty_PMH = df_frailty[frailty_cols]
df_frailty_PMH

In [ ]:
# To avoid SettingWithCopyWarning, make a copy before assignment
df_frailty_PMH = df_frailty_PMH.copy()

#calculate the frailty index by summing each other column and dividing by the number of available values for that patient, excluding the NaN values
df_frailty_PMH['frailty_available'] = ((df_frailty_PMH.notna().sum(axis=1)) - 1)
df_frailty_PMH['frailty_sum'] = ((df_frailty_PMH.iloc[:, 1:].sum(axis=1)) - (df_frailty_PMH['frailty_available']))
df_frailty_PMH['frailty_index_PMH'] = (df_frailty_PMH['frailty_sum'] / df_frailty_PMH['frailty_available']).round(2)

df_frailty_PMH['frailty_index_PMH_category'] = pd.cut(df_frailty_PMH['frailty_index_PMH'],
                                                         bins=[0, 0.1, 0.2, 0.55, 1], 
                                                         labels=['Fit', 'Pre-Frail', 'Frail', 'End-Stage Frail'],
                                                         right=False)

# Display the new DataFrame
df_frailty_PMH

In [ ]:
# Calculate the percentage of "1" (deficit) for columns 2 to 30 in df_frailty_PMH
start_col = 1  # 2nd column (index 1)
end_col = 42   # up to 43th column (index 42)
deficit_percentages = (
    (df_frailty_PMH.iloc[:, start_col:end_col] == 1).sum() /
    df_frailty_PMH.iloc[:, start_col:end_col].notna().sum() * 100
).round(1)

# Display as a DataFrame for readability
deficit_percentages_df = deficit_percentages.reset_index()
deficit_percentages_df.columns = ['Deficit', 'Percent with Deficit']
print(deficit_percentages_df)

#### 8b Frailty WITHOUT Past Medical History (29 variables)

In [ ]:
columns_to_keep =[
 'unique_episode',
# 'frailty_bmi',
 'frailty_SBP',
# 'frailty_heartRate',
 'frailty_dependency',
 'frailty_homeAssistance',
# 'frailty_DBP',
# 'frailty_bodyTemp',
 'frailty_SpO2',
 'frailty_sodium',
# 'frailty_CRP',
 'frailty_Hb',
 'frailty_WBC',
# 'frailty_PLT',
# 'frailty_AST',
# 'frailty_ALT',
# 'frailty_LDH',
 'frailty_Crea',
 'frailty_potassium',
# 'frailty_karnofsky',
# 'frailty_feeding',
# 'frailty_bathing',
# 'frailty_hygiene',
# 'frailty_dressing',
# 'frailty_bowel',
# 'frailty_bladder',
# 'frailty_toilet',
# 'frailty_transfers',
# 'frailty_walking',
# 'frailty_stairs'
]
df_frailty_noPMH = df_frailty[columns_to_keep]
df_frailty_noPMH


In [ ]:
# To avoid SettingWithCopyWarning, make a copy before assignment
df_frailty_noPMH = df_frailty_noPMH.copy()

#calculate the frailty index by summing each other column and dividing by the number of available values for that patient, excluding the NaN values
df_frailty_noPMH['frailty_available'] = ((df_frailty_noPMH.notna().sum(axis=1)) - 1)
df_frailty_noPMH['frailty_sum'] = ((df_frailty_noPMH.iloc[:, 1:].sum(axis=1)) - (df_frailty_noPMH['frailty_available']))
df_frailty_noPMH['frailty_index_noPMH'] = (df_frailty_noPMH['frailty_sum'] / df_frailty_noPMH['frailty_available']).round(2)

# Create categories for the frailty index as < 0.1, 0.1 to < 0.2, 0.2 to < 0.55, >= 0.55
df_frailty_noPMH['frailty_index_noPMH_category'] = pd.cut(
    df_frailty_noPMH['frailty_index_noPMH'], bins=[0, 0.1, 0.2, 0.55, 1], labels=['Fit', 'Pre-Frail', 'Frail', 'End-Stage Frail'],
    right=False)

# Display the new DataFrame
df_frailty_noPMH

#### 9 Test the characteristics of the frailty index/indices.

In [ ]:
# Calculate mean, standard deviation, median, Q1, and Q3 for frailty_index_PMH
pmh_mean = df_frailty_PMH['frailty_index_PMH'].mean()
pmh_std = df_frailty_PMH['frailty_index_PMH'].std()
pmh_median = df_frailty_PMH['frailty_index_PMH'].median()
pmh_q1 = df_frailty_PMH['frailty_index_PMH'].quantile(0.25)
pmh_q3 = df_frailty_PMH['frailty_index_PMH'].quantile(0.75)

# Calculate mean, standard deviation, median, Q1, and Q3 for frailty_index_noPMH
no_pmh_mean = df_frailty_noPMH['frailty_index_noPMH'].mean()
no_pmh_std = df_frailty_noPMH['frailty_index_noPMH'].std()
no_pmh_median = df_frailty_noPMH['frailty_index_noPMH'].median()
no_pmh_q1 = df_frailty_noPMH['frailty_index_noPMH'].quantile(0.25)
no_pmh_q3 = df_frailty_noPMH['frailty_index_noPMH'].quantile(0.75)

print(f"Frailty Index PMH: mean={pmh_mean:.2f}, sd={pmh_std:.2f}, median={pmh_median:.2f} (Q1={pmh_q1:.2f}, Q3={pmh_q3:.2f})")
print(f"Frailty Index noPMH: mean={no_pmh_mean:.2f}, sd={no_pmh_std:.2f}, median={no_pmh_median:.2f} (Q1={no_pmh_q1:.2f}, Q3={no_pmh_q3:.2f})")

In [ ]:
import seaborn as sns

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.kdeplot(df_frailty_PMH['frailty_index_PMH'].dropna(), label='FI including medical history', fill=False, alpha=1, color='#1f77b4')
sns.kdeplot(df_frailty_noPMH['frailty_index_noPMH'].dropna(), label='FI admission-based', fill=False, alpha=1, color='#ff7f0e')
#sns.kdeplot(df_frailty_chronic['frailty_index_chronic'].dropna(), label='Frailty Index (Chronic)', fill=False, alpha=0.5)
plt.xlabel('Frailty Index', fontsize=20)
plt.xticks(fontsize=18)
plt.xlim((0, 1)) # set x axis starting from 0 to 1 with steps of 0.2
plt.ylabel('Density', fontsize=22)
plt.legend(fontsize=18)
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Calculate value counts normalized to percentage for both categories
no_pmh_counts = df_frailty_noPMH['frailty_index_noPMH_category'].value_counts(normalize=True).sort_index() * 100
pmh_counts = df_frailty_PMH['frailty_index_PMH_category'].value_counts(normalize=True).sort_index() * 100

# Combine into a DataFrame for plotting
frailty_cat_df = pd.DataFrame({
    'No PMH': no_pmh_counts,
    'With PMH': pmh_counts
}).fillna(0)

# Plot as stacked bar
ax = frailty_cat_df.T.plot(kind='bar', stacked=True, color=["#8CE1A0", "#E0EB7E", "#DFCE82", "#EA8F8F"])
plt.ylabel('% of Total')
plt.title('Frailty Index Categories (No PMH vs With PMH)')
plt.legend(title='Frailty Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()

# Add percentage labels
for i, row in enumerate(frailty_cat_df.T.values):
    cum = 0
    for j, val in enumerate(row):
        if val > 0:
            ax.text(i, cum + val / 2, f"{val:.1f}", ha='center', va='center', fontsize=10, color='black')
            cum += val

plt.show()

#### 10. Use the frailty index in analyses.

### Export

In [ ]:
with pd.ExcelWriter(f'{output_directory}/CHOOSEFILENAME{pd.Timestamp.now().strftime("%Y%m%d")}.xlsx') as writer:
    df_frailty_PMH.to_excel(writer, sheet_name='df_frailty_withPMH', index=False)
    df_frailty_noPMH.to_excel(writer, sheet_name='df_frailty_noPMH', index=False)